# 0. Overview

The GT4Py V2 prototype: a minimal model of the GT4Py concepts, to try out the interface.

- `concepts.py`: the vocabulary, mostly `Protocol`s (structural interfaces: any object with the right members is an instance, without inheriting), type aliases, and a few helpers.
- `implementation.py`: simple dummy implementations, happy path only: numpy fields, decorators, and five backends.

New to the prototype? Start with [Quickstart L1: local semantics](quickstart_l1.ipynb), then [Quickstart L2: global semantics](quickstart_l2.ipynb).

Protocols are checked **statically**, by a type checker like mypy (names *and* types of the members), or **at runtime** with `isinstance` for the `@runtime_checkable` ones (only the *names*).

```
Python function --@field_operator() / @scan_operator(...)--> DSL operator (embedded execution)
DSL operator    --backend(operator), dispatching on its kind--> DSL operator of the backend
DSL operator    --called with fields and scalars, in a `with Mesh(...)` block if it uses relations--> Field
                  (op[domain](...): the result restricted to a requested domain)
local program   --run_local(program, local_meshes, args)--> one result per rank
Python function --@program()--> Program --lower(distribution)--> LoweredProgram (a local program per rank)

GTIRCompiler:   operator --frontend_transform--> GTIR --backend_transform--> compiled DSL operator
Allocation:     AllocatorProvider.__gt_allocator__ --> Allocator --> NDArray or NDArrayField
Backend:        a DSLInterpreter that is also an AllocatorProvider
```

After a change of `concepts.py` or `implementation.py`, restart the kernel of a notebook to reload them.

## The notebooks

Each notebook is self-contained, with its own examples:

| Notebook | Concepts | Examples |
|---|---|---|
| **Start here:** [Quickstart L1: local semantics](quickstart_l1.ipynb) | the L1 core of notebooks 1-11 in one walkthrough, a section per topic, each with a link to its notebook | heat diffusion along a row of 4 cells: fields, relations, operators, backends, local programs |
| **Then:** [Quickstart L2: global semantics](quickstart_l2.ipynb) | the L2 of notebooks 12-13 in one walkthrough: logical meshes and global programs | the same row as a logical mesh, with a global program lowered onto 1-3 ranks |
| [1. Fields and domains](01_fields_and_domains.ipynb) | dimensions, kinds, checked domains, index sets, `Field`, `NDArrayField`, containment, broadcasting, premap by offsets, values | temperatures on a grid, pressures on some cells, tracers of some cells |
| [2. Connectivities](02_connectivities.ipynb) | `Relation`, `Neighbor`, neighbor tables, `Mesh` and `with`, mesh extents, gathers, `sum_over`, `max_over`, `min_over`, local fields, domains through relations | a mesh of a quadrilateral and two triangles: perimeters, fluxes |
| [3. DSL definitions and operators](03_dsl_definitions_and_operators.ipynb) | definitions and DSL rules, forbidden names, `getsource`, field operators, extents, composite values, requested domains | diffusion, upwind advection, a hand-written operator |
| [4. Scan operators](04_scan_operators.ipynb) | scan definitions, `ScanConfig`, scan operators, scans in field operators, mesh columns | hydrostatic pressure, heights, decay |
| [5. Processors and backends](05_processors_and_backends.ipynb) | DSL processors, dispatch, the backends, lazy values, backward propagation, backends on a mesh, writing a backend | smoothing chains, a Laplacian on a ring of cells |
| [6. Compilation](06_compilation.ipynb) | compiled operators, `DSLCompiler`, `GTIRCompiler`, GTIR, transforms | kinetic energy, a cell mean, a constant-folding pass |
| [7. Allocation](07_allocation.ipynb) | arrays, allocators, field allocators, providers, memory layouts | column-major warming of columns |
| [8. Chains and partial relations](08_chains_and_partial_relations.ipynb) | neighbor ancestry, siblings, occurrences, partial relations, boundary values, read chains on the backends | heat across the edges of a triangle split in four |
| [9. Footprint algebra and pairs](09_footprint_algebra_and_pairs.ipynb) | footprint algebra, stencils, `reach`, derived tables, pairs | blending with an outside temperature, vertex interpolation with weights on pairs |
| [10. Local programs](10_local_programs.ipynb) | local meshes, decomposition, halos, regions, `Owned`, scatter/gather, local programs, `exchange`, `global_sum`, the simulation condition | mass sharing on a 4 × 2 plate, on 1-3 ranks |
| [11. Staggered axes](11_staggered_axes.ipynb) | `Staggered[D]`, `flip_staggered`, half offsets, operators and scans on half levels | interpolation between half and full levels |
| [12. Logical meshes](12_logical_meshes.ipynb) | schemas, `Face`/`Edge`/`Node`, roles, `LogicalMesh`, regions, geometry, `MeshField`, the brand, the signature | the plate as a logical mesh |
| [13. Global programs](13_global_programs.ipynb) | `@program`, global `sum_over`, `Distribution`, `distribute`, `derive_distribution`, coverage, the lowering, exchange placement, the simulation condition, the compile key | mass sharing on the plate, as in 10, without `Owned` and `exchange` |

[Semantics of domains, values and fields](semantics_of_domains_values_and_fields.ipynb) explains forward and backward domain computation, the order in which values are computed and when, and the functional patterns behind them (functor, comonad, Coyoneda), with plain numpy.

## Concepts and their implementations

| Concept (`concepts.py`) | Implemented in `implementation.py` by | Notebook |
|---|---|---|
| `DimensionIndex`, `Axis`, `Location`, `Label`, `Dimension`, `IndexSet`, `Domain`, `check_domain`, `Offset`, `Premap`, `Dims`, `Key` | `index_set`, `format_domain`; `NumPyArrayField` checks domains | [1](01_fields_and_domains.ipynb) |
| `Staggered`, `flip_staggered`, `HalfOffset` | half offsets in `NumPyArrayField` premaps and in `forward_domain`, `backward_domain` | [11](11_staggered_axes.ipynb) |
| `Field` (broadcasting), `NDArrayField`, `dims_of` | `NumPyArrayField` (1), `DeferredField` (5), `as_annotated` | [1](01_fields_and_domains.ipynb), [5](05_processors_and_backends.ipynb), [8](08_chains_and_partial_relations.ipynb) |
| `Scalar`, ..., `NamedCollection`, `AnyValue` | | [1](01_fields_and_domains.ipynb), [3](03_dsl_definitions_and_operators.ipynb) |
| `Relation` (`partial`), `is_relation`, `Neighbor[R, P]`, `neighbor_of`, `root_of`, `Boundary` | gathers, with reparenting and boundary values; `sum_over`, `max_over`, `min_over`, `neighbor_indices` | [2](02_connectivities.ipynb), [8](08_chains_and_partial_relations.ipynb) |
| `NeighborTable`, `SKIP_VALUE`, `MISSING_VALUE`, `check_neighbor_table` | `neighbor_table` | [2](02_connectivities.ipynb), [8](08_chains_and_partial_relations.ipynb) |
| `Mesh` (`extents`, `construct`, a context manager), `current_mesh` | used by gathers, `sum_over`, `neighbors`, `sources_within` | [2](02_connectivities.ipynb), [5](05_processors_and_backends.ipynb), [8](08_chains_and_partial_relations.ipynb), [9](09_footprint_algebra_and_pairs.ipynb) |
| `compose`, `Compose`, `Transpose`, `Expression`, `Stencil`, `composition_of`, `expression_name` | `paths_table`, `stencil_rows`, `construct_table` | [9](09_footprint_algebra_and_pairs.ipynb) |
| `Pairs`, `pairs_of`, `pair_relation` | `flatten`, `unflatten`, `pairs_rows` | [9](09_footprint_algebra_and_pairs.ipynb) |
| `InspectableCallable`, `CallableWithSource`, `getsource`, `call_operator`, `is_inspectable` | | [3](03_dsl_definitions_and_operators.ipynb) |
| `DSLOperatorDefinition`, `FieldOperatorDefinition`, `DSLRules` (`forbidden`, `violation`), `FIELD_OPERATOR_RULES`, `COLLECTIVES`, `is_*_definition` | plain functions, checked by `field_operator` | [3](03_dsl_definitions_and_operators.ipynb), [10](10_local_programs.ipynb) |
| `DSLOperatorKind`, `DSLOperator`, `FieldOperator` | `FieldOperatorWrapper` (from `@field_operator()`), `EagerFieldOperator`, `LazyFieldOperator`, `AllocatedFieldOperator` | [3](03_dsl_definitions_and_operators.ipynb), [5](05_processors_and_backends.ipynb) |
| `Extents`, `Step`, `Read`, `Reduction`, `ScanAlong`, `Flatten`, `Unflatten` | `read_extents`, `forward_domain`, `backward_domain` | [2](02_connectivities.ipynb), [3](03_dsl_definitions_and_operators.ipynb), [4](04_scan_operators.ipynb), [5](05_processors_and_backends.ipynb), [8](08_chains_and_partial_relations.ipynb), [9](09_footprint_algebra_and_pairs.ipynb) |
| `DomainSpec`, `restrict` | `DSLOperatorBase` (`op[domain]`, `domain=`) | [3](03_dsl_definitions_and_operators.ipynb) |
| `ScanOperatorDefinition`, `SCAN_OPERATOR_RULES`, `ScanConfig`, `ScanOperator` | `ScanOperatorWrapper` (from `@scan_operator(...)`), `EagerScanOperator`, `AllocatedScanOperator`, `GeneratedScanOperator` | [4](04_scan_operators.ipynb), [5](05_processors_and_backends.ipynb), [6](06_compilation.ipynb), [7](07_allocation.ipynb) |
| `DSLProcessor`, `dispatch`, `DSLInterpreter` | `eager`, `lazy`, `lazy_backward`, `column_major` | [5](05_processors_and_backends.ipynb) |
| `CompiledDSLOperator`, `CompiledFieldOperator`, `CompiledScanOperator` | `GeneratedOperator`, `GeneratedFieldOperator`, `GeneratedScanOperator` | [6](06_compilation.ipynb) |
| `DSLCompiler`, `GTIRCompiler`, `Transform`, `GTIR`, `definition_of` | `compiled` (`PythonCompiler`), `python_frontend`, `python_backend` | [6](06_compilation.ipynb) |
| `DType`, `Device`, `Shape`, `NDArray`, `NDArrayAllocator`, `ArrayNamespace` | `numpy` itself | [7](07_allocation.ipynb) |
| `NDArrayFieldAllocator`, `Allocator`, `AllocatorProvider` | `NumPyFieldAllocator`: `numpy_field_allocator`, `column_major_field_allocator`; `store` | [7](07_allocation.ipynb) |
| `Backend` | `eager`, `lazy`, `lazy_backward`, `column_major`, `compiled` (`BaseBackend` subclasses) | [5](05_processors_and_backends.ipynb), [6](06_compilation.ipynb), [7](07_allocation.ipynb) |
| `Region`, `LocalMesh`, `TRUNCATED`, `Owned`, `region` | `decompose`, `scatter`, `gather_owned`, `intersect`, `difference` | [10](10_local_programs.ipynb), [12](12_logical_meshes.ipynb), [13](13_global_programs.ipynb) |
| | `local_program`, `LocalProgram`, `run_local`, `exchange`, `global_sum` | [10](10_local_programs.ipynb) |
| `Schema`, `SchemaLocation`, `Face`, `Edge`, `Node`, `location_of`, `LogicalMesh`, `MeshField` | `NumPyMeshField`, `mesh_field` | [12](12_logical_meshes.ipynb) |
| `PROGRAM_RULES`, `program_mesh`, `Distribution` | `program`, `Program`, `LoweredProgram`, `distribute`, `derive_distribution`, `footprints`, `check_coverage` | [13](13_global_programs.ipynb) |

In [1]:
import hashlib
import pathlib

import concepts
import implementation

print("concepts:", len([n for n in vars(concepts) if not n.startswith("_")]), "public names")
print(
    "backends:",
    implementation.eager,
    implementation.lazy,
    implementation.lazy_backward,
    implementation.column_major,
    implementation.compiled,
)
for module in (concepts, implementation):  # to check the uploaded modules
    source = pathlib.Path(module.__file__).read_bytes()
    digest = hashlib.sha256(source).hexdigest()[:16]
    print(f"{module.__name__}.py: {len(source)} bytes, sha256 {digest}")

concepts: 145 public names
backends: eager lazy lazy_backward column_major compiled
concepts.py: 100435 bytes, sha256 4e6edb61945850be
implementation.py: 156174 bytes, sha256 91f11c23b6f340da
